# Lab 1 - Intrinsic Explainable Models

**How the lab works**

- The questions are divided into three levels.
  - **[Basic]**: if your group completes all Basic questions of a lab correctly, you get a 6 (a passing grade).
  - **[Regular]**: these questions give up to 2 extra points.
  - **[Advanced]**: for students looking for a challenge; expect to work on them in your own time.
- Every group works with slightly different data and settings, so your answers, numbers and plots are different from those of other groups.
  - Set your group number $G$ in the first code cell. Your seed is $S = G$; use it everywhere a random seed is needed (`random_state=S`).
  - *Your patient* is row $p = S \bmod n$ of your test set, where $n$ is the number of rows in the test set.
  - Put your group number in the title of every plot.
- Answers must use the numbers and plots from your own notebook. An answer that does not match your own output gets no points.
- You may use AI tools to help with code, but every group member must be able to explain what you hand in.
- Lines marked `# TODO` must be completed. Write your answers in the **Your answers** cells.
- Hand in this notebook with all cells run.

**Data.** We want to explain the classification of the target class (heart disease or not) from the 13 attributes:
`age`, `sex`, `cp` (chest pain type), `trestbps` (resting blood pressure), `chol`, `fbs`, `restecg`,
`thalach` (maximum heart rate), `exang` (exercise-induced angina), `oldpeak`, `slope`, `ca`, `thal`.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

from xai_lab_utils import group_seed, personal_data_path, your_patient_index

In [ ]:
# Fill in your group number. Everything else depends on it.
GROUP_NUMBER = 19  # TODO: your group number, e.g. 7

S = group_seed(GROUP_NUMBER)   # use S everywhere a random seed is needed
print("Group", GROUP_NUMBER, "- seed S =", S)

## Q1) Load the data

a) **[Basic]** Load the data file of your group (`personal_data_path`). What is the file format? Which method of `pandas` reads this format? Which values of `sep`, `header` and `decimal` does your file need, and why?

b) **[Basic]** For each of the 13 attributes, say if it is numerical, binary or categorical. Give your answer as a table.

c) **[Regular]** Load the file once more with a wrong value for `sep` or `decimal`. What happens to the columns? How can you see the problem with `df.info()`?

*Note:* to be able to explain AI methods, it is also important to have good data analysis skills. If the data is imported incorrectly, we are trying to explain meaningless results.

> **Hint:** `pd.read_csv(path, sep=..., header=..., decimal=...)`; `df.info()`, `df.describe()`, `df.nunique()`

In [ ]:
path = personal_data_path(GROUP_NUMBER)
print(path)

# Look at the raw file first: which separator and decimal sign does it use?
with open(path, encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().rstrip())

In [ ]:
df = pd.read_csv(path, sep=",", header=0, decimal=".")

df.info()
df.head()

**Your answers:**

a) It's a normal CSV, loaded with `pd.read_csv`. It needs `sep=","`, `header=0` and `decimal="."`, since that's what the raw file actually uses (commas between values, header on the first line, dots for decimals in `oldpeak`). These match the pandas defaults, so it loads fine, but other groups might have `;` or `,` instead, so we still set them explicitly to be sure.

b) 

c) 

## Q2) Perform basic exploratory data analysis

a) **[Basic]** Before you compute anything, write down the three attributes you expect to be most related to heart disease. Then: if you were limited to 5 of the 13 attributes, how would you choose them without a trained model? Show the result as a heatmap of the correlation matrix.

b) **[Basic]** Which 5 attributes have the strongest correlation with the target in your data? Give the values. Was your expectation right?

c) **[Basic]** How does this help to explain the model later? What does correlation *not* tell us?

d) **[Regular]** Correlation is not a good measure for categorical attributes such as `cp` and `thal`. Explain why. Compute the mutual information of each attribute with the target and compare the ranking with the correlation ranking.

e) **[Advanced]** Find the two attributes that are most correlated with each other (not with the target). Train a logistic model with both attributes, and then with only one of them. How do the coefficients change? Explain why.

> **Hint:** `plt.figure(figsize=(10, 8))`; `corr = df.corr()`; `sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)`;
> `sklearn.feature_selection.mutual_info_classif(X, y, discrete_features=mask, random_state=S)`

In [ ]:
plt.figure(figsize=(10, 8))
corr = None  # TODO: compute the correlation matrix with a pandas method
# TODO: show it with sns.heatmap(...)
plt.title(f"Correlation matrix - group {GROUP_NUMBER}")
plt.show()

**Your answers:**

a) 

b) 

c) 

d) 

e) 

## Q3) Train intrinsic explainable models

a) **[Basic]** Complete the preprocessing below (which attributes are scaled, one-hot encoded or kept as they are). Train a logistic model, a decision tree and a kNN model. Report the accuracy and ROC-AUC of each model on your test set.

b) **[Basic]** Why do we scale the attributes for the logistic model and kNN, but not for the decision tree?

> **Hint:** `model.fit(X_train, y_train)`, `model.predict(X_test)`, `model.predict_proba(X_test)[:, 1]`; `accuracy_score`, `roc_auc_score`

In [ ]:
X = df.drop(columns="target").astype(float)   # float: needed for PDP in recent scikit-learn
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=S)

# TODO: put every attribute in one of the three lists (use your table from Q1 b)
num_cols = []   # numerical attributes: will be scaled
cat_cols = []   # categorical attributes: will be one-hot encoded
bin_cols = []   # binary attributes: kept as they are

preprocessor = ColumnTransformer(
    [("num", StandardScaler(), num_cols),
     ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), cat_cols),
     ("bin", "passthrough", bin_cols)],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

In [ ]:
logreg = make_pipeline(clone(preprocessor), LogisticRegression(max_iter=1000))
tree = DecisionTreeClassifier(random_state=S)            # a tree needs no scaling
knn = make_pipeline(clone(preprocessor), KNeighborsClassifier(n_neighbors=5))

models = {"logistic": logreg, "tree": tree, "kNN": knn}

# TODO: fit each model on the training data and report accuracy and ROC-AUC on the test set
for name, model in models.items():
    pass

In [ ]:
# Your patient: row p of your test set
p = your_patient_index(GROUP_NUMBER, len(X_test))
x_p = X_test.iloc[[p]]
print("Your patient is row", p, "of the test set")
x_p

**Your answers:**

a) 

b) 

## Q4) Explaining the logistic model

a) **[Basic]** List the parameters that were fit in the logistic model and explain what they represent.

b) **[Basic]** Compute the odds ratio $e^{\beta}$ of each attribute. For the three most important attributes, write one sentence each, using your own numbers. Example: "if `thalach` increases by one standard deviation, the odds of heart disease are multiplied by ...".

c) **[Basic]** Compute the predicted probability for your patient by hand, using the intercept, the coefficients and the scaled values of your patient. Show the calculation and compare it with `predict_proba`.

d) **[Advanced]** Train the model once on unscaled and once on scaled data. Compare the order of the attributes by $|\beta|$. Why can we not compare coefficients of unscaled data?

e) **[Advanced]** Train the model with `cp` as a number (0, 1, 2, 3) and with `cp` one-hot encoded. How does the explanation of `cp` change? Which version makes sense?

f) **[Advanced]** Use `statsmodels` to get 95% confidence intervals for the odds ratios. Which intervals contain 1? What does that mean for your explanation?

> **Hint:** `clf.coef_`, `clf.intercept_`, `np.exp(clf.coef_)`; scaled values of your patient: `logreg[:-1].transform(x_p)`;
> `import statsmodels.api as sm`; `res = sm.Logit(y, sm.add_constant(X)).fit()`; `np.exp(res.conf_int())`

In [ ]:
clf = logreg[-1]                              # the fitted LogisticRegression
names = logreg[:-1].get_feature_names_out()   # attribute names after preprocessing
coef = pd.Series(clf.coef_[0], index=names)
print("intercept:", clf.intercept_[0])
coef.sort_values()

In [ ]:
# TODO (b): odds ratios
# TODO (c): probability of your patient by hand. The scaled values of your patient are:
logreg[:-1].transform(x_p)

**Your answers:**

a) 

b) 

c) 

d) 

e) 

f) 

## Q5) Explaining the decision tree

a) **[Basic]** Plot the trained tree so that a person can use it to explain a decision. Follow the path of your patient through the tree and write it as one if-then rule.

b) **[Regular]** Train trees with the split methods `"gini"`, `"entropy"` and `"log_loss"`. Plot the accuracies and comment on how the trees differ.

c) **[Regular]** Take the class counts at the root node of your tree. Compute the Gini index, the entropy and the classification error of the root split by hand. Show the calculation.

d) **[Regular]** Plot the train and test accuracy for `max_depth` from 1 to 12. Which depth would you show to a doctor? Explain your choice.

e) **[Advanced]** Train a tree with `max_depth=3` on five bootstrap samples of your training data (seeds $S, S+1, \dots, S+4$). Compare the first two levels of the trees. What does this say about the stability of tree explanations?

> **Hint:** `plot_tree(tree, feature_names=..., class_names=..., filled=True, max_depth=3)`, `export_text(tree, feature_names=...)`;
> `DecisionTreeClassifier(criterion="entropy", random_state=S)`; `tree.decision_path(x_p)`;
> `sklearn.utils.resample(X_train, y_train, random_state=S + i)`

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(tree, feature_names=list(X.columns), class_names=["no disease", "disease"],
          filled=True, max_depth=3, fontsize=9)
plt.title(f"Decision tree (top 3 levels) - group {GROUP_NUMBER}")
plt.show()

# Nodes visited by your patient:
print(tree.decision_path(x_p).indices)

In [ ]:
# TODO (b): train trees with criterion "gini", "entropy" and "log_loss" and plot the accuracies
for criterion in ["gini", "entropy", "log_loss"]:
    pass

In [ ]:
# Class counts at the root node (for the calculation by hand in (c)).
# In recent scikit-learn versions tree_.value holds class fractions, so multiply by the node size.
n_root = tree.tree_.n_node_samples[0]
print("root counts:", np.round(tree.tree_.value[0][0] * n_root))

# TODO (d): train and test accuracy for max_depth = 1, ..., 12

**Your answers:**

a) 

b) 

c) 

d) 

e) 

## Q6) Explaining kNN

a) **[Basic]** Plot the accuracy for different values of $k$ (odd values from 1 to 51). Comment on how the explanation changes with different values of $k$.

b) **[Basic]** Look at the $k$ nearest neighbours of your patient in the training data (code below). Would this convince a doctor? Why or why not?

c) **[Regular]** Repeat b) without scaling. Which attribute now decides who the neighbours are? Why?

d) **[Advanced]** Find the smallest change in one attribute of your patient that changes the prediction (a counterfactual). Is this change realistic? You may compare with `dice-ml`.

> **Hint:** `knn.set_params(kneighborsclassifier__n_neighbors=k)`;
> `dice_ml.Dice(data, model).generate_counterfactuals(x_p, total_CFs=3, desired_class="opposite")`

In [ ]:
# TODO (a): accuracy for odd k from 1 to 51
ks = range(1, 52, 2)

In [ ]:
# (b) The k nearest neighbours of your patient
knn_clf = knn[-1]
dist, idx = knn_clf.kneighbors(knn[:-1].transform(x_p), n_neighbors=knn_clf.n_neighbors)
neighbours = X_train.iloc[idx[0]].assign(target=y_train.iloc[idx[0]].values, distance=dist[0])
neighbours

**Your answers:**

a) 

b) 

c) 

d) 

## Q7) Reflection

a) **[Basic]** Use the decision tree to explain the prediction for your patient (i) to the patient and (ii) to a doctor. What is different between the two explanations?

b) **[Regular]** Assess the decision tree on completeness, comprehensibility and stability, based on what you saw in this lab.

**Your answers:**

a) 

b) 